# Upgrades

My paper ended with changes it recommended. I found four bugs while rebuilding it. Here I measure both next to my original analysis. v1 is my original analysis as I reproduced it. v2 adds four fixes, two features measured per day and new lines for the rule.

Every number here comes from `reports/upgrades.json`. `python -m bleedtracker upgrades` builds that file from the warehouse. It first rebuilds my original results and checks them against `reports/results.json`.

In [1]:
import pandas as pd

from bleedtracker.upgrades import load_upgrades

upgrades = load_upgrades()
deployments = upgrades["deployments"]


def side_by_side(queue, measure):
    """One column per version for one queue and one measure."""
    return pd.DataFrame(
        {
            f"{queue} {version}": {d["deployment"]: d[version][queue][measure] for d in deployments}
            for version in ["v1", "v2"]
        }
    )

## Post-hoc and in-sample

I built every upgrade after I had opened the sealed month. None of it was written down before. Every result here is post-hoc.

Rule v2's slide line and riser gate were picked on the training clients of the four open month pairs. The held-out clients played no part in the pick. The months they're judged on were part of it. v2's results on the open months are partly in-sample.

The sealed month played no part in the pick. It's still post-hoc. I chose to build v2 after I had seen it.

In [2]:
s = upgrades["settings"]
pd.Series(
    {
        "v2 slide line": s["v2_slide_line"],
        "v2 riser gate": s["v2_riser_gate"],
        "A7 line": s["a7_psi"],
        "bootstrap draws": f"{s['bootstrap_draws']:,}",
    },
    name="setting",
).to_frame()

,setting
v2 slide line,0.65
v2 riser gate,1.2
A7 line,0.25
bootstrap draws,"10,000"


The pick moved the slide line from 0.8 to 0.65. It couldn't move the riser gate. The rule scores a rising page zero. The forest ranks one too low for its shadow queue. The gate stays at 1.20.

## The four fixes

Each fix is counted on its own against my original version.

In [3]:
fixes = upgrades["fixes"]
pd.Series(fixes["empty_second_half_pages"], name="pages whose rows stopped mid-month").to_frame()

,pages whose rows stopped mid-month
2026-01,56
2026-02,49
2026-03,100
2026-04,83
2026-05,235


**Empty second half.** My original code scored a page whose rows stopped mid-month as flat. It also kept that page out of the queue. The fix gives it momentum 0. That touched between 49 and 235 pages a month. In the sealed feature month it was 235 of 82,570.

In [4]:
pd.DataFrame(fixes["coverage_verify_first"]).set_index("deployment").rename_axis(
    "editor slots marked verify first"
)

,original,fixed
editor slots marked verify first,,
anchor 2026-03 to 2026-04,13,13
forward 2026-02 to 2026-03,23,23
forward 2026-03 to 2026-04,13,13
forward 2026-04 to 2026-05,13,13
sealed 2026-05 to 2026-06,5,5


**Coverage.** My original gate compared a page's live days, as a share of its month, with 25/31. The fixed gate counts 25 days. The two differ only in a 28-day month. February was the only one. No queued page sat in the gap. The verify-first items stayed the same in every deployment.

In [5]:
pd.DataFrame(fixes["a1"]).set_index("release").rename_axis("A1 population moved")

,original,fixed
A1 population moved,,
2026-02 to 2026-03,False,False
2026-03 to 2026-04,True,True
2026-04 to 2026-05,False,False
2026-05 to 2026-06,False,False


**A1.** No release moved by exactly 20%. The fix changed no alarm.

In [6]:
pd.DataFrame(fixes["queue_csv_reasons"]).fillna(0).astype(int).rename_axis(
    "pages in the sealed CSV"
)

,original,fixed
pages in the sealed CSV,,
holding_steady,19175,9259
durable_riser,7223,7223
new_or_no_first_half_traffic,96,96
slipping_and_underclicking,45,3644
quiet_risk_model_only,10,10
measurement_suspicion,5,5
slipping_reachable_page,0,4574
slipping_deep_page,0,1743


**Reasons in the queue CSV.** In the sealed month, 9,916 pages were sliding below the cut. My original CSV called each of them "holding_steady". The fixed CSV gives each one its rule reason. 4,574 are reachable, 3,599 are underclicking and 1,743 are deep.

## v1 against v2, margin first

v2 opens the sealed month through its own pre-release gate. That gate uses the fixed A1.

In [7]:
gate = upgrades["sealed_gate_v2"]
pd.Series(
    {
        "decision": gate["decision"],
        "pages against the month before": gate["pages_vs_previous_month"],
        "worst PSI against training": gate["worst_psi"],
    },
    name="v2's gate on the sealed month",
).to_frame()

,v2's gate on the sealed month
decision,RELEASE
pages against the month before,1.0375
worst PSI against training,0.1459


The gate released it. The worst drift against training was position at 0.146. v1's gate read the same.

In [8]:
pd.concat([side_by_side("rule", "margin_points"), side_by_side("forest", "margin_points")], axis=1)

,rule v1,rule v2,forest v1,forest v2
anchor 2026-03 to 2026-04,40.6,40.6,40.6,42.6
forward 2026-02 to 2026-03,68.6,64.6,58.6,58.6
forward 2026-03 to 2026-04,40.6,40.6,48.6,48.6
forward 2026-04 to 2026-05,57.2,57.2,55.2,55.2
sealed 2026-05 to 2026-06,24.4,24.4,26.4,26.4


Rule v2 didn't beat rule v1. Their margins match in four deployments. In February, rule v2 fell 4 points. Its margin went from 68.6 to 64.6.

Forest v2 gained 2 points in the anchor deployment. It matched forest v1 everywhere else.

On the sealed month, both rules put 49 declining pages in their 50.

## The share of lost impressions

Each queue's pages went on to lose some of the impressions the whole pool lost. A random 50 would hold about 0.2% of them.

In [9]:
shares = pd.concat(
    [
        side_by_side("rule", "lost_impressions_share"),
        side_by_side("forest", "lost_impressions_share"),
    ],
    axis=1,
)
shares["random 50"] = side_by_side("rule", "random_lost_impressions_share")["rule v1"]
shares

,rule v1,rule v2,forest v1,forest v2,random 50
anchor 2026-03 to 2026-04,0.0225,0.0206,0.0037,0.0042,0.0019
forward 2026-02 to 2026-03,0.0191,0.0181,0.0363,0.0291,0.0024
forward 2026-03 to 2026-04,0.0225,0.0206,0.0007,0.0006,0.0019
forward 2026-04 to 2026-05,0.0293,0.0244,0.0045,0.0046,0.0020
sealed 2026-05 to 2026-06,0.0230,0.0206,0.0055,0.0059,0.0019


The rule's 50 pages hold between 1.8% and 2.9% of the pool's lost impressions. The forest's 50 hold less than the rule's in every deployment but February.

Rule v2 holds less than rule v1 in every deployment. I haven't checked why.

## The per-account queue

Slots follow each held-out client's share of the pool's pages. Every client gets at least one. The rule picks from eligible pages. The forest picks from the whole pool. Both follow how their precision@50 is built.

In [10]:
per_account = pd.concat(
    [side_by_side("rule", "margin_points"), side_by_side("per_account_rule", "margin_points")],
    axis=1,
)
per_account["slots per client"] = [
    str(d["v1"]["per_account_rule"]["slots_per_client"]) for d in deployments
]
per_account

,rule v1,rule v2,per_account_rule v1,per_account_rule v2,slots per client
anchor 2026-03 to 2026-04,40.6,40.6,34.6,36.6,"[1, 1, 1, 1, 1, 2, 3, 4, 36]"
forward 2026-02 to 2026-03,68.6,64.6,65.4,62.6,"[1, 1, 1, 1, 2, 3, 4, 37]"
forward 2026-03 to 2026-04,40.6,40.6,34.6,36.6,"[1, 1, 1, 1, 1, 2, 3, 4, 36]"
forward 2026-04 to 2026-05,57.2,57.2,57.2,53.2,"[1, 1, 1, 1, 2, 2, 2, 3, 37]"
sealed 2026-05 to 2026-06,24.4,24.4,18.7,20.7,"[1, 1, 1, 1, 2, 2, 3, 39]"


In the sealed month the largest client took 39 of the 50 slots. Spreading slots over clients cost the rule up to 6 points of margin. In April it cost rule v1 nothing.

## The bootstrap intervals

Each interval resamples one deployment's held-out pages 10,000 times. Both top 50s are rebuilt on every draw. The gap is the forest minus the rule, in points.

In [11]:
pd.DataFrame(
    {
        "v1 gap": {
            d["deployment"]: d["v1"]["bootstrap"]["forest_minus_rule_points"] for d in deployments
        },
        "v1 90% interval": {
            d["deployment"]: d["v1"]["bootstrap"]["interval_90"] for d in deployments
        },
        "v2 gap": {
            d["deployment"]: d["v2"]["bootstrap"]["forest_minus_rule_points"] for d in deployments
        },
        "v2 90% interval": {
            d["deployment"]: d["v2"]["bootstrap"]["interval_90"] for d in deployments
        },
    }
)

,v1 gap,v1 90% interval,v2 gap,v2 90% interval
anchor 2026-03 to 2026-04,0.0,"[-8.0, 10.0]",2.0,"[-6.0, 12.0]"
forward 2026-02 to 2026-03,-10.0,"[-24.0, 2.0]",-6.0,"[-20.0, 4.0]"
forward 2026-03 to 2026-04,8.0,"[0.0, 16.0]",8.0,"[2.0, 18.0]"
forward 2026-04 to 2026-05,-2.0,"[-8.0, 4.0]",-2.0,"[-8.0, 4.0]"
sealed 2026-05 to 2026-06,2.0,"[0.0, 6.0]",2.0,"[0.0, 6.0]"


Only forest v2 in the forward March to April deployment has an interval above zero. It runs from 2 to 18 points. Forest v1's interval there and both sealed ones start at 0. Every other interval includes zero.

These intervals treat pages as independent. Pages of one client share a template, a CMS and a calendar. The uncertainty is wider than these intervals show.

## A7 against the month before

A7 compares each release with the same clients one month earlier. It reads active days per day.

In [12]:
a7 = pd.DataFrame(upgrades["a7"]["releases"]).set_index("release")
pd.concat(
    [
        pd.DataFrame(list(a7.psi_against_previous_month), index=a7.index),
        a7.A7_drifted_since_last_month,
    ],
    axis=1,
)

,log_impressions,active_days,position,ctr,momentum,A7_drifted_since_last_month
release,,,,,,
2026-02 to 2026-03,0.0219,0.0475,0.1158,0.0064,0.0540,False
2026-03 to 2026-04,0.0036,0.0658,0.0714,0.0048,0.2841,True
2026-04 to 2026-05,0.0019,0.0014,0.0636,0.0042,0.0553,False
2026-05 to 2026-06,0.0049,0.0026,0.4224,0.0149,0.0395,True


A7 fires twice. On the sealed release, position read 0.4224. From March to April, momentum read 0.2841. My original A2 stayed quiet on both. A7 warns and never holds a release.